In [43]:
import numpy as np
import pandas as pd
import json

from sklearn.ensemble import RandomForestRegressor



from scipy.stats import wasserstein_distance, spearmanr

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr
import torch


import random

In [44]:
with open("../data/enviroment_bbox.json", "r") as f:
    bbox = json.load(f)

enviroment_bbox = [
    bbox["min_lon"],
    bbox["min_lat"],
    bbox["max_lon"],
    bbox["max_lat"]
]

In [ ]:
from dataset_loader import DataBuilder
save = False
sp = "SQA"
features = ["TO", "SO", "TOB", 
            "MLOTST", "UGO", "VGO", "ZO",
            "CHL", "PP", "CDM", "SPM", "ZSD",
            "MSEN", "MCOS",
            ]
todas = False

data_builder = DataBuilder(target_var="cpue_index", sp=sp, features=features, 
                           prediction_horizon=1, forecast_lead=1, window_size=12,
                           start_date="2012-01-01", end_date="2025-12-31",
                            enviroment_bbox=enviroment_bbox, split_year=2024, split_year_test=2025,
                            )

vars_names = data_builder.features
(X_train, y_train, m_train), (X_val, y_val, m_val), (X_test, y_test, m_test) = data_builder.get_splits()

def prep_ml_data(X, y, m=None):
    """
    Converts 5D input tensors and 3D target tensors into 2D arrays for ML
    """
    # Move to CPU and convert to numpy if dealing with PyTorch Tensors
    if isinstance(X, torch.Tensor):
        X = X.detach().cpu().numpy()
    if isinstance(y, torch.Tensor):
        y = y.detach().cpu().numpy()
    if isinstance(m, torch.Tensor):
        m = m.detach().cpu().numpy()

    N = X.shape[0]
    
    # Flatten features: (N, T, C, H_x, W_x) -> (N, T * C * H_x * W_x)
    X_rf = X.reshape(N, -1)
    
    # Flatten targets: (N, H_y, W_y) -> (N, H_y * W_y)
    y_rf = y.reshape(N, -1)
    
    if m is not None:
        m_rf = m.reshape(N, -1)
        return X_rf, y_rf, m_rf
    else:
        return X_rf, y_rf

X_train_ml, y_train_ml, m_train_ml = prep_ml_data(X_train, y_train, m_train)
X_val_ml, y_val_ml, m_val_ml = prep_ml_data(X_val, y_val, m_val)
X_test_ml, y_test_ml, m_test_ml = prep_ml_data(X_test, y_test, m_test)



prediction_horizon = y_test.shape[1]
H_val, W_val = y_val.shape[-2], y_val.shape[-1]
H_out, W_out = y_test.shape[-2], y_test.shape[-1]

m_true_val = m_val_ml.reshape(-1, prediction_horizon, H_val, W_val)
y_true_val = y_val_ml.reshape( -1, prediction_horizon, H_val, W_val)

m_true = m_test_ml.reshape(-1, prediction_horizon, H_out, W_out)
y_true = y_test_ml.reshape(-1, prediction_horizon, H_out, W_out)

mask_flat = m_test_ml.astype(bool)


# data_builder.plot_correlations()

Loading and aligning dataset...
Preprocessing and scaling...
Creating rolling windows...
Train: X torch.Size([144, 12, 15, 54, 49]) | y torch.Size([144, 1, 30, 17])
Val:   X torch.Size([12, 12, 15, 54, 49]) | y torch.Size([12, 1, 30, 17])
Test:  X torch.Size([12, 12, 15, 54, 49]) | y torch.Size([12, 1, 30, 17])


In [46]:
def compute_metrics(y_test_ml, y_pred_flat, y_true, mask_spatial):


    N, horizon, H_out, W_out = y_true.shape

    mask_spatial = np.asarray(mask_spatial).astype(bool)

    # =====================================================
    # 1. GLOBAL FLATTENED METRICS
    # =====================================================

    y_test_flat = np.asarray(y_test_ml).flatten()
    y_pred_flat_1d = np.asarray(y_pred_flat).flatten()
    mask_flat = mask_spatial.flatten()

    y_test_masked = y_test_flat[mask_flat]
    y_pred_masked = y_pred_flat_1d[mask_flat]

    # =====================================================
    # 2. REGRESSION METRICS
    # =====================================================

    rmse = np.sqrt(mean_squared_error(y_test_masked, y_pred_masked))
    mae = mean_absolute_error(y_test_masked,y_pred_masked)
    r2 = r2_score(y_test_masked,y_pred_masked)
    pearson_corr = pearsonr(y_test_masked,y_pred_masked)[0]
    mbe = np.mean(y_pred_masked - y_test_masked)
    data_range_flat = (np.max(y_test_masked)- np.min(y_test_masked))
    
    if data_range_flat > 0:
        nrmse = rmse / data_range_flat
    else:
        nrmse = np.nan    
        
    w_dist = wasserstein_distance(y_test_masked, y_pred_masked)
        
    spearman_corr, p_val = spearmanr(y_test_masked,y_pred_masked)

    # =====================================================
    # 3. RESTORE 4D SHAPE
    # =====================================================

    y_pred4D = np.asarray(y_pred_flat).reshape(N, horizon, H_out, W_out)

    # =====================================================
    # 5. Per month metrics
    # =====================================================
    for h in range(horizon):
        y_true_h = y_true[:, h, :, :]
        y_pred_h = y_pred4D[:, h, :, :]
        mask_h = mask_spatial[:, h, :, :]

        # Flatten automatically by boolean masking
        y_true_h_masked = y_true_h[mask_h]
        y_pred_h_masked = y_pred_h[mask_h]

        if len(y_true_h_masked) > 1:
            r2_h = r2_score(y_true_h_masked, y_pred_h_masked)
        else:
            r2_h = np.nan

        print(f"Month +{h + 1}: R² = {r2_h:.4f}", f"{y_pred_h_masked.shape}")
    print("-----------------------------------------------------------------------")
    # =====================================================
    # 6. RETURN
    # =====================================================
    metrics = [r2, pearson_corr, spearman_corr, rmse, nrmse, mae, mbe, w_dist, ]


    return metrics

In [47]:
iters = 2
columns = ["r2", "pearson_corr", "spearman_corr", "rmse", "nrmse", "mae", "mbe", "w_dist"]
metrics_df = pd.DataFrame(columns=columns)
importance_all_df = pd.DataFrame(columns=vars_names)

for i in range(iters):
    model = RandomForestRegressor(n_estimators=200, max_features="sqrt", n_jobs=-1, random_state=random.randint(0, 1000))

    model.fit(X_train_ml, y_train_ml)

    y_pred_flat = model.predict(X_test_ml)

    metrics = compute_metrics(y_test_ml, y_pred_flat, y_true, m_true)
    metrics_df.loc[len(metrics_df)] = metrics


    # -------- Feature Importance Extraction --------
    raw_importances = model.feature_importances_
    T = X_train.shape[1]   # window (12)
    C = X_train.shape[2]   # number of variables/channels
    H_x = X_train.shape[3] # height (lat)
    W_x = X_train.shape[4] # width (lon)

    reshaped_importances = raw_importances.reshape(T, C, H_x, W_x)

    variable_importances = reshaped_importances.sum(axis=(0, 2, 3))


    importance_all_df.loc[len(importance_all_df)] = variable_importances


importance_summary_df = pd.DataFrame({"Variable": vars_names, 
                                      "Importance": [f"{importance_all_df[var].mean():.3f}±{importance_all_df[var].std():.3f}"for var in vars_names],
                                      "Mean Importance": [importance_all_df[var].mean()for var in vars_names]})

# Sort by mean importance
importance_summary_df = importance_summary_df.sort_values(by="Mean Importance",ascending=False).reset_index(drop=True)

# Optional: remove helper column before saving
importance_summary_df = importance_summary_df.drop(columns=["Mean Importance"])

metrics_summary_df = pd.DataFrame(
{col: [f"{metrics_df[col].mean():.3f}±{metrics_df[col].std():.3f}"]for col in metrics_df.columns})


    
display(metrics_summary_df)
display(importance_summary_df)


Month +1: R² = 0.5400 (1500,)
-----------------------------------------------------------------------
Month +1: R² = 0.5207 (1500,)
-----------------------------------------------------------------------


,r2,pearson_corr,spearman_corr,rmse,nrmse,mae,mbe,w_dist
0,0.530±0.014,0.738±0.013,0.633±0.003,1.215±0.018,0.106±0.002,0.729±0.007,-0.026±0.002,0.494±0.013


,Variable,Importance
0,PP,0.304±0.005
1,TO,0.097±0.008
2,MLOTST,0.086±0.000
3,SPM,0.062±0.002
4,CHL,0.061±0.001
5,ZSD,0.060±0.003
6,ZO,0.050±0.002
7,UGO,0.049±0.003
8,VGO,0.049±0.001
9,CDM,0.048±0.003


In [48]:
if save:    
# save everyhtng to a excel workbook
    model_name = model.__class__.__name__
    sheet_name = f"{model_name}_{sp}"
    if todas:
        sheet_name = f"{model_name}_{sp}_todas"

    with pd.ExcelWriter("./modelos/resultados/Regre/ML_reg.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        metrics_summary_df.to_excel(writer, sheet_name=sheet_name, startrow=1, index=False)

    with pd.ExcelWriter("./modelos/resultados/Regre/ML_reg.xlsx", engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
        importance_summary_df.to_excel(writer, sheet_name=sheet_name, startrow=len(metrics_summary_df) + 3, index=False)

### Anomaly prediction model

In [49]:
from dataset_loader import DataBuilder

sp = "SQA"

features = ["TO", "SO", "TOB", "MLOTST", "UGO", "VGO", "ZO", "CHL", "PP", "CDM", "SPM", "ZSD", "MSEN", "MCOS"]

todas = False

data_builder = DataBuilder(target_var="cpue_index", sp=sp, features=features, prediction_horizon=1, forecast_lead=1, window_size=12, 
                           start_date="2012-01-01", end_date="2025-12-31", enviroment_bbox=enviroment_bbox, anomaly=True,
                           split_year=2024, split_year_test=2025)

vars_names = data_builder.features

(X_train, y_train, m_train), (X_val, y_val, m_val), (X_test, y_test, m_test) = data_builder.get_splits()

X_train_ml, y_train_ml, m_train_ml = prep_ml_data(X_train, y_train, m_train)
X_val_ml, y_val_ml, m_val_ml = prep_ml_data(X_val, y_val, m_val)
X_test_ml, y_test_ml, m_test_ml = prep_ml_data(X_test, y_test, m_test)

#extract anomaly and original targets and reshape them
y_val_original = data_builder.y_val_original.detach().cpu().numpy()
y_test_original = data_builder.y_test_original.detach().cpu().numpy()

clim_val = data_builder.clim_val.detach().cpu().numpy()
clim_test = data_builder.clim_test.detach().cpu().numpy()

y_val_original_ml = y_val_original.reshape(y_val_original.shape[0], -1)
y_test_original_ml = y_test_original.reshape(y_test_original.shape[0], -1)

clim_val_ml = clim_val.reshape(clim_val.shape[0], -1)
clim_test_ml = clim_test.reshape(clim_test.shape[0], -1)


prediction_horizon = y_test.shape[1]
H_val, W_val = y_val.shape[-2], y_val.shape[-1]
H_out, W_out = y_test.shape[-2], y_test.shape[-1]

m_true_val = m_val_ml.reshape(-1, prediction_horizon, H_val, W_val)
y_true_val = y_val_original_ml.reshape(-1, prediction_horizon, H_val, W_val)

m_true = m_test_ml.reshape(-1, prediction_horizon, H_out, W_out)
y_true = y_test_original_ml.reshape(-1, prediction_horizon, H_out, W_out)

mask_flat = m_test_ml.astype(bool)

Loading and aligning dataset...
Preprocessing and scaling...
Creating rolling windows...
Train: X torch.Size([144, 12, 14, 54, 49]) | y torch.Size([144, 1, 30, 17])
Val:   X torch.Size([12, 12, 14, 54, 49]) | y torch.Size([12, 1, 30, 17])
Test:  X torch.Size([12, 12, 14, 54, 49]) | y torch.Size([12, 1, 30, 17])


In [50]:
iters = 2
columns = ["r2", "pearson_corr", "spearman_corr", "rmse", "nrmse", "mae", "mbe", "w_dist"]
metrics_df = pd.DataFrame(columns=columns)
importance_all_df = pd.DataFrame(columns=vars_names)

for i in range(iters):
    model = RandomForestRegressor(n_estimators=200, max_features="sqrt", n_jobs=-1, random_state=random.randint(0, 1000))

    model.fit(X_train_ml, y_train_ml)

    y_pred_anomaly_flat = model.predict(X_test_ml)

    y_pred_flat =clim_test_ml + y_pred_anomaly_flat  


    metrics = compute_metrics(y_test_original_ml, y_pred_flat, y_true, m_true)
    metrics_df.loc[len(metrics_df)] = metrics


    # -------- Feature Importance Extraction --------
    raw_importances = model.feature_importances_
    T = X_train.shape[1]   # window (12)
    C = X_train.shape[2]   # number of variables/channels
    H_x = X_train.shape[3] # height (lat)
    W_x = X_train.shape[4] # width (lon)

    reshaped_importances = raw_importances.reshape(T, C, H_x, W_x)

    variable_importances = reshaped_importances.sum(axis=(0, 2, 3))


    importance_all_df.loc[len(importance_all_df)] = variable_importances


importance_summary_df = pd.DataFrame({"Variable": vars_names, 
                                      "Importance": [f"{importance_all_df[var].mean():.3f}±{importance_all_df[var].std():.3f}"for var in vars_names],
                                      "Mean Importance": [importance_all_df[var].mean()for var in vars_names]})

# Sort by mean importance
importance_summary_df = importance_summary_df.sort_values(by="Mean Importance",ascending=False).reset_index(drop=True)

# Optional: remove helper column before saving
importance_summary_df = importance_summary_df.drop(columns=["Mean Importance"])

metrics_summary_df = pd.DataFrame(
{col: [f"{metrics_df[col].mean():.3f}±{metrics_df[col].std():.3f}"]for col in metrics_df.columns})

display(metrics_summary_df)
display(importance_summary_df)

Month +1: R² = 0.4524 (1500,)
-----------------------------------------------------------------------
Month +1: R² = 0.4454 (1500,)
-----------------------------------------------------------------------


,r2,pearson_corr,spearman_corr,rmse,nrmse,mae,mbe,w_dist
0,0.449±0.005,0.671±0.003,0.602±0.007,1.316±0.006,0.114±0.001,0.750±0.007,-0.072±0.011,0.437±0.012


,Variable,Importance
0,SPM,0.118±0.005
1,CHL,0.099±0.004
2,ZSD,0.094±0.001
3,ZO,0.091±0.001
4,VGO,0.080±0.002
5,UGO,0.079±0.000
6,MLOTST,0.076±0.001
7,TO,0.076±0.003
8,PP,0.074±0.005
9,SO,0.059±0.000
